# Week 3 — Numerical Computing with NumPy

**Internship:** AI/ML Internship · 24-Week Practical Task Plan  
**Assignment:** Perform numerical analysis on a dataset using NumPy.

## Objectives
- Work with NumPy arrays, indexing, slicing, and broadcasting.
- Perform at least **10 numerical operations**.
- Perform at least **5 statistical calculations**.
- Perform at least **3 matrix operations**.
- Explain the results in plain language.

> **Dataset note:** `retail_sales.csv` is a deterministic, synthetic teaching dataset created for this assignment. It is not real company or customer data. Amounts are in Indian rupees (INR).

## 1. Import libraries and load the dataset

The built-in `csv` module reads the file; NumPy arrays are used for the numerical analysis.

In [1]:
from pathlib import Path
import csv
import json
import numpy as np

# The notebook is designed to run from the project root or from the notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "retail_sales.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "retail_sales.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

with DATA_PATH.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    records = list(reader)

print("Dataset loaded successfully")
print("Number of records:", len(records))
print("Columns:", list(records[0].keys()))

Dataset loaded successfully
Number of records: 120
Columns: ['order_id', 'month', 'product', 'region', 'units_sold', 'unit_price_inr', 'discount_pct', 'marketing_spend_inr', 'returns', 'customer_rating']


In [2]:
# Convert the numeric columns to NumPy arrays.
month = np.array([int(row["month"]) for row in records])
units = np.array([float(row["units_sold"]) for row in records])
price = np.array([float(row["unit_price_inr"]) for row in records])
discount_pct = np.array([float(row["discount_pct"]) for row in records])
marketing = np.array([float(row["marketing_spend_inr"]) for row in records])
returns = np.array([float(row["returns"]) for row in records])
rating = np.array([float(row["customer_rating"]) for row in records])

print("units array shape:", units.shape)
print("units array data type:", units.dtype)
print("Feature sample (first 5 units):", units[:5])

units array shape: (120,)
units array data type: float64
Feature sample (first 5 units): [25. 30. 32. 20. 36.]


## 2. Array indexing and slicing

- **Indexing** reads one element using its position.
- **Slicing** selects a range of elements using `start:stop:step`.

In [3]:
print("First record units sold:", units[0])
print("Records 1 to 5 (positions 0–4):", units[0:5])
print("Every second record among the first 10:", units[0:10:2])
print("Last three records:", units[-3:])

First record units sold: 25.0
Records 1 to 5 (positions 0–4): [25. 30. 32. 20. 36.]
Every second record among the first 10: [25. 32. 36. 35. 49.]
Last three records: [50. 63. 64.]


## 3. Numerical operations (14 examples)

The calculations below cover element-wise arithmetic, safe division, transformations, conditional selection, aggregation, and cumulative calculations.

In [4]:
# Operation 1: element-wise multiplication — gross sales before discount.
gross_revenue = units * price

# Operation 2: element-wise multiplication — discount value.
discount_amount = gross_revenue * discount_pct

# Operation 3: subtraction — revenue after discount.
net_revenue = gross_revenue - discount_amount

# Operation 4: safe element-wise division — revenue per unit.
revenue_per_unit = np.divide(net_revenue, units, out=np.zeros_like(net_revenue), where=units != 0)

# Operation 5: calculate return rate per record.
return_rate = np.divide(returns, units, out=np.zeros_like(returns), where=units != 0)

# Operation 6: standard-score transformation (z-score).
revenue_z_score = (net_revenue - np.mean(net_revenue)) / np.std(net_revenue)

# Operation 7: min-max normalization of units sold.
units_minmax = (units - np.min(units)) / (np.max(units) - np.min(units))

# Operation 8: conditional categorization using np.where.
sales_band = np.where(net_revenue >= np.percentile(net_revenue, 75), "High",
              np.where(net_revenue >= np.percentile(net_revenue, 25), "Medium", "Low"))

# Operation 9: concatenate two arrays into one longer array.
first_half = net_revenue[:len(net_revenue)//2]
second_half = net_revenue[len(net_revenue)//2:]
combined_revenue = np.concatenate((first_half, second_half))

# Operation 10: reshape a 1-D feature array into a 2-D array (12 months x 10 records).
monthly_units_matrix = units.reshape(12, 10)

# Operation 11: month-wise weighted aggregation with np.bincount.
monthly_revenue = np.bincount(month, weights=net_revenue, minlength=13)[1:13]

# Operation 12: month-to-month difference.
monthly_revenue_change = np.diff(monthly_revenue)

# Operation 13: cumulative revenue over the year.
cumulative_revenue = np.cumsum(monthly_revenue)

# Operation 14: constrain ratings to the valid 1–5 scale.
rating_clipped = np.clip(rating, 1, 5)

print("1. Gross revenue sample:", np.round(gross_revenue[:3], 2))
print("2. Discount amount sample:", np.round(discount_amount[:3], 2))
print("3. Net revenue sample:", np.round(net_revenue[:3], 2))
print("4. Revenue per unit sample:", np.round(revenue_per_unit[:3], 2))
print("5. Return rate sample:", np.round(return_rate[:3], 4))
print("6. Revenue z-score sample:", np.round(revenue_z_score[:3], 3))
print("7. Normalized units sample:", np.round(units_minmax[:3], 3))
print("8. Sales bands sample:", sales_band[:8])
print("9. Concatenated array length:", len(combined_revenue))
print("10. Reshaped monthly units matrix:", monthly_units_matrix.shape)
print("11. Monthly revenue total:", round(float(np.sum(monthly_revenue)), 2))
print("12. Monthly revenue change sample:", np.round(monthly_revenue_change[:3], 2))
print("13. Cumulative revenue at year end:", round(float(cumulative_revenue[-1]), 2))
print("14. Ratings remain within 1–5:", bool(np.all((rating_clipped >= 1) & (rating_clipped <= 5))))

1. Gross revenue sample: [16015.25 19942.2  56844.16]
2. Discount amount sample: [1601.52 1994.22    0.  ]
3. Net revenue sample: [14413.72 17947.98 56844.16]
4. Revenue per unit sample: [ 576.55  598.27 1776.38]
5. Return rate sample: [0.08   0.0667 0.0312]
6. Revenue z-score sample: [-1.549 -1.42   0.007]
7. Normalized units sample: [0.102 0.204 0.245]
8. Sales bands sample: ['Low' 'Low' 'Medium' 'Low' 'Medium' 'Low' 'Medium' 'Medium']
9. Concatenated array length: 120
10. Reshaped monthly units matrix: (12, 10)
11. Monthly revenue total: 6797493.68
12. Monthly revenue change sample: [71177.     710.73 25037.9 ]
13. Cumulative revenue at year end: 6797493.68
14. Ratings remain within 1–5: True


## 4. Broadcasting

Broadcasting lets NumPy apply compatible arrays of different shapes without writing an explicit Python loop. Here, one adjustment factor per month is applied to every record belonging to that month.

In [5]:
monthly_adjustment_factors = np.linspace(0.95, 1.06, 12)  # 12 monthly factors
record_factors = monthly_adjustment_factors[month - 1]  # one factor mapped to each record
adjusted_revenue = net_revenue * record_factors

# Column-wise standardization also demonstrates broadcasting: a (4,) mean/std
# vector is applied to each row of the (120, 4) feature matrix.
features = np.column_stack((units, price, discount_pct, marketing))
feature_mean = np.mean(features, axis=0)
feature_std = np.std(features, axis=0)
feature_std_safe = np.where(feature_std == 0, 1, feature_std)
standardized_features = (features - feature_mean) / feature_std_safe

print("Monthly factor array shape:", monthly_adjustment_factors.shape)
print("Mapped factor array shape:", record_factors.shape)
print("Adjusted revenue sample:", np.round(adjusted_revenue[:5], 2))
print("Standardized feature matrix shape:", standardized_features.shape)
print("Standardized column means (approximately zero):", np.round(np.mean(standardized_features, axis=0), 8))

Monthly factor array shape: (12,)
Mapped factor array shape: (120,)
Adjusted revenue sample: [13693.04 17050.58 54001.95 31710.82 40117.68]
Standardized feature matrix shape: (120, 4)
Standardized column means (approximately zero): [-0.  0.  0. -0.]


## 5. Statistical calculations (at least five)

The assignment requires at least five statistical calculations. We calculate mean, median, variance, standard deviation, and the 90th percentile. Minimum and maximum are also included for context.

In [6]:
statistics = {
    "Mean": np.mean(net_revenue),
    "Median": np.median(net_revenue),
    "Population variance": np.var(net_revenue),
    "Population standard deviation": np.std(net_revenue),
    "90th percentile": np.percentile(net_revenue, 90),
    "Minimum": np.min(net_revenue),
    "Maximum": np.max(net_revenue),
}
for name, value in statistics.items():
    print(f"{name}: INR {value:,.2f}")

Mean: INR 56,645.78
Median: INR 53,138.69
Population variance: INR 743,075,283.33
Population standard deviation: INR 27,259.41
90th percentile: INR 97,192.61
Minimum: INR 12,133.08
Maximum: INR 136,070.96


## 6. Matrix operations (more than three)

A feature matrix is formed from units sold, unit price, discount percentage, and marketing spend. Its columns are standardized before matrix calculations so that the different units/scales do not dominate the demonstration.

In [7]:
# Matrix operation 1: transpose a matrix.
feature_transpose = standardized_features.T
print("Original feature matrix:", standardized_features.shape)
print("Transposed feature matrix:", feature_transpose.shape)

# Matrix operation 2: matrix multiplication to produce a 4 x 4 Gram matrix.
gram_matrix = feature_transpose @ standardized_features
print("Gram matrix shape:", gram_matrix.shape)
print(np.round(gram_matrix, 3))

# Matrix operation 3: covariance matrix across the four standardized features.
covariance_matrix = np.cov(standardized_features, rowvar=False)
print("Covariance matrix shape:", covariance_matrix.shape)
print(np.round(covariance_matrix, 4))

# Additional matrix operations: determinant, inverse, matrix-vector multiplication, solve.
regularized_covariance = covariance_matrix + np.eye(covariance_matrix.shape[0]) * 1e-8
determinant = np.linalg.det(regularized_covariance)
inverse_matrix = np.linalg.inv(regularized_covariance)
coefficients = np.array([0.40, 0.25, -0.10, 0.25])
weighted_activity_index = standardized_features @ coefficients
linear_system_solution = np.linalg.solve(regularized_covariance, np.ones(4))

print("Regularized covariance determinant:", f"{determinant:.10f}")
print("Inverse matrix shape:", inverse_matrix.shape)
print("Matrix-vector product sample:", np.round(weighted_activity_index[:5], 3))
print("Linear-system solution:", np.round(linear_system_solution, 4))

Original feature matrix: (120, 4)
Transposed feature matrix: (4, 120)
Gram matrix shape: (4, 4)
[[120.      6.512  -4.822  94.416]
 [  6.512 120.    -10.608  12.685]
 [ -4.822 -10.608 120.      0.21 ]
 [ 94.416  12.685   0.21  120.   ]]
Covariance matrix shape: (4, 4)
[[ 1.0084  0.0547 -0.0405  0.7934]
 [ 0.0547  1.0084 -0.0891  0.1066]
 [-0.0405 -0.0891  1.0084  0.0018]
 [ 0.7934  0.1066  0.0018  1.0084]]
Regularized covariance determinant: 0.3835746784
Inverse matrix shape: (4, 4)
Matrix-vector product sample: [-1.756 -1.37  -0.664 -1.194 -0.687]
Linear-system solution: [0.7535 1.0185 1.1115 0.2892]


## 7. Monthly summary and key observations

Monthly revenue is aggregated with `np.bincount`. `np.argmax` and `np.argmin` locate the months with the highest and lowest revenue.

In [8]:
month_labels = np.arange(1, 13)
best_month = int(month_labels[np.argmax(monthly_revenue)])
lowest_month = int(month_labels[np.argmin(monthly_revenue)])

print("Month | Net revenue (INR) | Month-over-month change (INR)")
for i, value in enumerate(monthly_revenue):
    change = "—" if i == 0 else f"{monthly_revenue_change[i-1]:,.2f}"
    print(f"{i+1:>5} | {value:>18,.2f} | {change:>30}")

print("\nHighest-revenue month:", best_month)
print("Lowest-revenue month:", lowest_month)
print("Overall net revenue (INR):", f"{np.sum(net_revenue):,.2f}")
print("Total units sold:", int(np.sum(units)))
print("Overall return rate:", f"{np.sum(returns)/np.sum(units):.2%}")

Month | Net revenue (INR) | Month-over-month change (INR)
    1 |         421,891.40 |                              —
    2 |         493,068.40 |                      71,177.00
    3 |         493,779.13 |                         710.73
    4 |         518,817.03 |                      25,037.90
    5 |         509,636.62 |                      -9,180.41
    6 |         529,051.33 |                      19,414.71
    7 |         590,458.56 |                      61,407.22
    8 |         590,111.57 |                        -346.99
    9 |         639,598.61 |                      49,487.04
   10 |         635,577.99 |                      -4,020.62
   11 |         706,580.49 |                      71,002.49
   12 |         668,922.55 |                     -37,657.94

Highest-revenue month: 11
Lowest-revenue month: 1
Overall net revenue (INR): 6,797,493.68
Total units sold: 5305
Overall return rate: 4.41%


## 8. Save analysis outputs

The notebook saves machine-readable summary results and monthly totals in the `outputs/` folder so the calculations can be reviewed outside Jupyter.

In [9]:
summary = {
    "dataset": "Synthetic retail sales dataset",
    "record_count": int(len(units)),
    "total_units_sold": int(np.sum(units)),
    "total_gross_revenue_inr": float(np.sum(gross_revenue)),
    "total_discount_amount_inr": float(np.sum(discount_amount)),
    "total_net_revenue_inr": float(np.sum(net_revenue)),
    "overall_return_rate": float(np.sum(returns) / np.sum(units)),
    "highest_revenue_month": best_month,
    "lowest_revenue_month": lowest_month,
    "statistics": {k: float(v) for k, v in statistics.items()},
    "matrix_shapes": {
        "features": list(standardized_features.shape),
        "transpose": list(feature_transpose.shape),
        "gram": list(gram_matrix.shape),
        "covariance": list(covariance_matrix.shape),
    },
}
(OUTPUT_DIR / "summary_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
with (OUTPUT_DIR / "monthly_revenue.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["month", "net_revenue_inr", "month_over_month_change_inr", "cumulative_revenue_inr"])
    for i in range(12):
        writer.writerow([i+1, round(float(monthly_revenue[i]), 2),
                         "" if i == 0 else round(float(monthly_revenue_change[i-1]), 2),
                         round(float(cumulative_revenue[i]), 2)])
np.savetxt(OUTPUT_DIR / "correlation_matrix.csv", np.corrcoef(features, rowvar=False), delimiter=",", fmt="%.6f")
print("Saved summary_metrics.json, monthly_revenue.csv, and correlation_matrix.csv")

Saved summary_metrics.json, monthly_revenue.csv, and correlation_matrix.csv


## 9. Conclusion

This exercise demonstrates how NumPy arrays can make numerical work concise and reproducible. The analysis calculates revenue and return-related measures, normalizes and classifies values, aggregates monthly totals, computes descriptive statistics, and performs matrix transformations and multiplication. Because the data is synthetic, the numeric findings are used to demonstrate the methods and should not be interpreted as actual business performance.